<div style="display:flex; justify-content:space-between; align-items:center; width:100%; margin:8px 0 24px 0;"><div style="text-align:left;"><img src="https://www.ec-nantes.fr/medias/photo/logocn-rvb_1648479844750-png?ID_FICHE=178994&amp;INLINE=FALSE" alt="Centrale Nantes" style="height:72px; width:auto;"></div><div style="text-align:right; font-size:18px; font-weight:600; color:#17324d; line-height:1.35;">MSc. CORO DASSIP</div></div>
<div style="border:2px solid #333; padding:14px 20px; margin:15px auto 25px auto; width:85%; max-width:900px; box-sizing:border-box; text-align:center;"><h1 style="margin:0;"><b>Filtering in Spatial Domain — Algorithm</b></h1></div>


## Purpose and Reading Convention

This notebook explains how to implement and review this lab inside its existing `labs/` directory. The executable code remains in [main.ipynb](main.ipynb); no repository split or Python package extraction is required.

Each numbered step gives **Purpose**, **Inputs**, **Procedure**, **Outputs**, **Validation** and **On failure**. Follow dependencies in order and reuse the same intermediate arrays for metrics and figures.

**Validation convention:** these are review contracts. Some are enforced by existing assertions or exceptions; others require inspecting intermediate results. A documentation update does not add runtime guards. The *Implementation limits* section identifies important gaps so the algorithm does not claim unimplemented protections.

Use parameter values and numeric tolerances actually declared in `main.ipynb`. Do not replace them with unsupported universal thresholds. A failed implementation check stops the run; a diagnostic limitation must be reported rather than silently treated as a pass.

## Inputs and Final Outputs

| Item | Description |
| --- | --- |
| **Inputs** | Reference, noisy, blurred, and RGB lab images |
| **Final outputs** | Spatial-filtering comparisons, denoising/sharpening/gradient diagnostics, 18 required figures |
| **Implementation target** | `main.ipynb` |

## 0. Start

**Purpose**

Establish the scope and a clean execution order.

**Inputs**

The problem statement, theory, requirements and main notebook in this lab.

**Procedure**

1. Select the project environment and use this notebook as the coding sequence for `main.ipynb`.
2. Keep boundary handling, source images, and comparison parameters explicit.
3. Use clean references whenever quantitative denoising metrics are computed.

**Outputs**

Selected environment and a top-to-bottom execution plan.

**Validation**

Review the local assumptions before using cached notebook outputs.

**On failure**

Resolve missing prerequisites first; do not infer a successful run from stored outputs.

## 1. Create Paths and Discover Input Images

**Purpose**

Associate each experiment with a known input.

**Inputs**

Lab data files and semantic role patterns.

**Procedure**

1. Import `Path`, math, NumPy, Matplotlib, PIL, SciPy `ndimage`, and OpenCV.
2. Locate the lab root and create data/output paths.
3. Discover supported files recursively.
4. Map semantic roles such as clean reference, Gaussian-noisy, salt-and-pepper, speckle, blurred moon, and RGB examples to deterministic files.

**Outputs**

INPUT_IMAGES and output directory.

**Validation**

Every clean/noisy role resolves deterministically.

**On failure**

Stop on missing data; do not silently pair unrelated images.

## 2. Load and Validate Images

**Purpose**

Establish image and channel conventions.

**Inputs**

Resolved paths.

**Procedure**

1. Implement grayscale and RGB image loaders.
2. Load each semantic-role image exactly once.
3. Print shape, dtype, and intensity range for each image.
4. Stop if any required role cannot be resolved.

**Outputs**

Grayscale/RGB uint8 arrays.

**Validation**

Nonempty arrays; corresponding clean/noisy dimensions agree.

**On failure**

Correct decoding or role matching before comparisons.

## 3. Create Toy Arrays and Reference Kernels

**Purpose**

Make local filtering arithmetic inspectable.

**Inputs**

Small known image and kernels.

**Procedure**

1. Create small numeric arrays for hand-verifiable filtering examples.
2. Define a normalized mean kernel and an asymmetric kernel.
3. Define Sobel and Laplacian kernels explicitly.
4. Use float arithmetic for kernel computations.

**Outputs**

Toy neighborhoods and reference kernels.

**Validation**

Odd support; smoothing kernels sum to one where intended.

**On failure**

Do not normalize derivative kernels as if they were averages.

## 4. Differentiate Correlation from Convolution

**Purpose**

Expose the kernel flip that distinguishes convolution.

**Inputs**

Asymmetric kernel and toy image.

**Procedure**

1. Select one local patch from the toy array.
2. Compute correlation using the asymmetric kernel unchanged.
3. Compute convolution after flipping the kernel in both axes.
4. Compare manual center values with SciPy `correlate` and `convolve`.

**Outputs**

Manual correlation/convolution values and SciPy references.

**Validation**

Compare the same centre pixel; flip both kernel axes only for convolution.

**On failure**

Fix kernel orientation if manual and reference values differ.

## 5. Implement Direct 2-D Convolution

**Purpose**

Implement the neighborhood operation explicitly.

**Inputs**

2-D image and odd-size kernel.

**Procedure**

1. Create a helper accepting image and odd-sized kernel.
2. Compute half-kernel padding sizes.
3. Pad the image symmetrically.
4. Flip the kernel to implement convolution.
5. Loop over rows/columns, extract each neighborhood, multiply, sum, and store the result.

**Outputs**

Same-shaped floating convolution result.

**Validation**

Symmetric padding; kernel flipped in two axes; no uint8 accumulation.

**On failure**

Reject even kernel sizes under the current helper's convention.

## 6. Validate Manual Convolution Against SciPy

**Purpose**

Verify the manual implementation against a library.

**Inputs**

Manual result and ndimage.convolve with equivalent boundaries.

**Procedure**

1. Convert toy image and kernel to float64.
2. Run the manual implementation and SciPy convolution with equivalent boundary behaviour.
3. Compute maximum absolute difference.
4. Assert agreement to tight floating-point tolerance.

**Outputs**

Numerical equality check.

**Validation**

rtol=1e-10 and atol=1e-12 for this floating toy comparison.

**On failure**

Inspect padding and flip conventions before changing tolerance.

## 7. Demonstrate Border Modes

**Purpose**

Show how missing neighbors are synthesized.

**Inputs**

One image/kernel and several border modes.

**Procedure**

1. Create a controlled image with content near the boundary.
2. Apply the same filter under different padding conventions.
3. Plot with identical display limits.
4. Use the comparison to reveal edge artefacts.

**Outputs**

Border-mode comparison.

**Validation**

Kernel unchanged; only boundary convention varies.

**On failure**

Document the mode instead of treating border differences as algorithm errors.

## 8. Apply Mean Filtering

**Purpose**

Establish a uniform smoothing baseline.

**Inputs**

Image and averaging kernel.

**Procedure**

1. Create a normalized 3×3 box kernel.
2. Convolve the Gaussian-noisy reference image using reflective boundaries.
3. Compare clean, noisy, and filtered images.
4. Save the baseline mean-filter result.

**Outputs**

Mean-filtered image.

**Validation**

Kernel sums to one; source shape preserved.

**On failure**

Investigate brightness shifts from an unnormalized kernel.

## 9. Study Mean-Kernel Size

**Purpose**

Measure the smoothing/detail tradeoff of support size.

**Inputs**

Same source and several odd window sizes.

**Procedure**

1. Repeat mean filtering for sizes 3, 5, and 9.
2. Change only kernel size.
3. Store outputs in a dictionary keyed by size.
4. Save the controlled comparison.

**Outputs**

Mean-filter sweep.

**Validation**

Same border mode and source; label each size.

**On failure**

Do not change noise realization during a support-size comparison.

## 10. Build a 2-D Gaussian Kernel

**Purpose**

Create a normalized finite Gaussian kernel.

**Inputs**

Odd size and positive sigma.

**Procedure**

1. Require positive sigma and odd kernel size.
2. Create centered x/y coordinate grids.
3. Evaluate the Gaussian response.
4. Normalize to unit sum and verify the sum is approximately one.

**Outputs**

Floating kernel shaped (size,size).

**Validation**

Nonnegative, symmetric weights with sum near one.

**On failure**

Existing assertions reject even size/nonpositive sigma.

## 11. Verify Gaussian Separability

**Purpose**

Verify a Gaussian factorization on the same support.

**Inputs**

1-D and 2-D Gaussian kernels.

**Procedure**

1. Build the equivalent normalized 1-D Gaussian vector.
2. Compute its outer product.
3. Compare that product with the explicit 2-D Gaussian kernel.
4. Assert numerical equivalence.

**Outputs**

Outer-product comparison.

**Validation**

np.allclose confirms separability for matching size/sigma.

**On failure**

Align truncation and normalization before comparing kernels.

## 12. Apply Gaussian Filtering and Sigma Sweep

**Purpose**

Relate sigma to blur strength.

**Inputs**

Same image and chosen sigma values.

**Procedure**

1. Filter the Gaussian-noisy image with a representative sigma.
2. Evaluate several sigma values while keeping image and boundary handling fixed.
3. Store outputs by sigma.
4. Save the comparison to expose smoothing-versus-detail trade-off.

**Outputs**

Gaussian outputs and sweep figure.

**Validation**

Fixed source/boundary settings; record sigma per output.

**On failure**

Check settings rather than attributing all changes to window size.

## 13. Compare Mean and Gaussian Smoothing

**Purpose**

Compare different weighting within a neighborhood.

**Inputs**

Mean and Gaussian outputs on the same image.

**Procedure**

1. Apply comparable mean and Gaussian smoothing to the same noisy input.
2. Plot noisy input and both filtered results together.
3. Keep display scaling identical.
4. Defer method selection until quantitative metrics are available.

**Outputs**

Paired smoothing diagnostic.

**Validation**

Common source, display range and documented support.

**On failure**

Do not claim equal blur solely because support sizes match.

## 14. Inspect the Noise Models

**Purpose**

Identify the corruption model before choosing a filter.

**Inputs**

Clean, Gaussian, impulse and speckle examples.

**Procedure**

1. Display clean reference, Gaussian noise, salt-and-pepper noise, and speckle noise side by side.
2. Use these known noise types to motivate filter choice.
3. Do not compare filters across different noise inputs as if the conditions were identical.

**Outputs**

Noise-model overview.

**Validation**

Correct clean/noisy pairing and matching dimensions.

**On failure**

Resolve mismatched files before computing reference metrics.

## 15. Apply Median Filtering to Impulse Noise

**Purpose**

Test a rank-based filter on impulse corruption.

**Inputs**

Salt-and-pepper image and odd median window.

**Procedure**

1. Demonstrate mean versus median on a small sample containing an outlier.
2. Apply a 3×3 median filter to the salt-and-pepper image.
3. Compare mean, Gaussian, and median outputs on the same impulse-noise input.
4. Save the comparison.

**Outputs**

Median-filtered image.

**Validation**

Source shape preserved; display removed impulses and lost detail.

**On failure**

Do not interpret median filtering as linear convolution.

## 16. Study Median Window Size

**Purpose**

Observe the detail loss from larger median windows.

**Inputs**

Same noisy image and several odd windows.

**Procedure**

1. Evaluate median sizes 3, 5, and 7 on the same salt-and-pepper image.
2. Store outputs keyed by size.
3. Compare impulse removal against detail loss.
4. Save the size sweep.

**Outputs**

Median sweep.

**Validation**

Only window size varies.

**On failure**

Inspect thin-structure removal before choosing the largest window.

## 17. Apply Bilateral Filtering

**Purpose**

Use spatial and intensity distance jointly.

**Inputs**

Image; bilateral diameter, sigmaColor and sigmaSpace.

**Procedure**

1. Apply OpenCV bilateral filtering to the Gaussian-noisy image.
2. Set neighborhood diameter, `sigmaColor`, and `sigmaSpace` explicitly.
3. Compare reference, noisy input, Gaussian smoothing, and bilateral output.
4. Keep the source image fixed.

**Outputs**

Bilateral-filtered output.

**Validation**

Parameters expressed in the input intensity/spatial units.

**On failure**

Correct dtype or intensity scale before tuning sigmaColor.

## 18. Study Bilateral Parameters

**Purpose**

Distinguish bilateral parameter effects.

**Inputs**

Fixed source and declared parameter combinations.

**Procedure**

1. Vary `sigmaColor` across the declared values.
2. Keep diameter and `sigmaSpace` fixed.
3. Store and plot each result.
4. Use the sweep to isolate photometric-similarity effects.

**Outputs**

Bilateral sweep.

**Validation**

Record each combination; vary parameters in controlled comparisons.

**On failure**

Do not attribute a multi-parameter change to one parameter alone.

## 19. Implement Denoising Metrics

**Purpose**

Quantify denoising against matched clean data.

**Inputs**

Reference and same-shaped estimate.

**Procedure**

1. Create MAE, MSE, RMSE, and PSNR helpers in float arithmetic.
2. Return infinite PSNR only when MSE is exactly zero.
3. Create one metric-summary helper so every method is scored identically.

**Outputs**

MAE, MSE and PSNR.

**Validation**

Float differences; zero MSE yields infinite PSNR.

**On failure**

Reject misaligned references; report expected infinite self-PSNR correctly.

## 20. Compare Filters on Gaussian Noise

**Purpose**

Compare filters on additive Gaussian corruption.

**Inputs**

Clean reference and one noisy image.

**Procedure**

1. Score noisy input, mean, Gaussian, and bilateral outputs against the clean reference.
2. Print one aligned metric table.
3. Use the same reference and peak value for every method.
4. Retain results for later method selection.

**Outputs**

Per-filter outputs and metric table.

**Validation**

All scores use the same reference and noise realization.

**On failure**

Inspect pairing/ranges if scores contradict displayed inputs.

## 21. Compare Filters on Salt-and-Pepper Noise

**Purpose**

Compare filters on sparse impulse corruption.

**Inputs**

Clean reference and salt-and-pepper image.

**Procedure**

1. Score noisy input, mean, Gaussian, and median outputs against the same clean reference.
2. Print the same metric set and formatting.
3. Keep the noise model fixed while changing only the filter.

**Outputs**

Per-filter outputs and metrics.

**Validation**

Same input/reference; preserve parameter labels.

**On failure**

Do not generalize the ranking to all noise models.

## 22. Measure Edge Preservation

**Purpose**

Add a detail diagnostic to reference error.

**Inputs**

Clean/noisy/filtered images.

**Procedure**

1. Implement Sobel gradient magnitude.
2. Compute gradient maps for the clean reference and denoised candidates.
3. Compare mean gradient magnitude as a secondary detail-retention diagnostic.
4. Use this together with pixel-error metrics.

**Outputs**

Mean gradient-magnitude summaries.

**Validation**

Compare like-for-like ranges; noise can also raise gradients.

**On failure**

Do not equate a larger gradient score with better denoising.

## 23. Sharpen with the Laplacian

**Purpose**

Enhance local contrast using second derivatives.

**Inputs**

Image and the declared Laplacian kernel.

**Procedure**

1. Convolve the blurred moon with the explicit Laplacian kernel.
2. Combine original and Laplacian according to the chosen sign convention.
3. Clip only the final sharpened image to `[0,255]`.
4. Save input, Laplacian response, and sharpened output.

**Outputs**

Laplacian response and sharpened result.

**Validation**

Use floating/signed response; sharpening sign follows kernel convention.

**On failure**

Correct sign or overflow before clipping the displayed result.

## 24. Implement Unsharp and High-Boost Filtering

**Purpose**

Separate low-frequency content from detail.

**Inputs**

Image, blur and gain.

**Procedure**

1. Gaussian-blur the moon to estimate its low-frequency component.
2. Compute detail mask = original − blurred.
3. Create unsharp output with gain 1 and high-boost output with a larger gain.
4. Clip final outputs and save the comparison.

**Outputs**

Detail mask and sharpened images.

**Validation**

detail=image-blur; image+gain*detail; gain zero preserves source.

**On failure**

Keep signed detail; clip only display/storage output.

## 25. Compute First-Derivative Gradients

**Purpose**

Measure directional intensity changes.

**Inputs**

Grayscale image and derivative operators.

**Procedure**

1. Compute horizontal and vertical Sobel derivatives.
2. Compute gradient magnitude and orientation.
3. Visualize components, magnitude, and orientation consistently.
4. Keep derivative arrays in float until display.

**Outputs**

Gx, Gy, magnitude and orientation.

**Validation**

Magnitude=sqrt(Gx**2+Gy**2); orientation=atan2(Gy,Gx).

**On failure**

Correct coordinate/sign conventions; do not erase negative derivatives.

## 26. Compare Prewitt, Sobel, and Scharr

**Purpose**

Compare derivative discretizations.

**Inputs**

Same image and Prewitt/Sobel/Scharr operators.

**Procedure**

1. Apply each operator to the same float image.
2. Combine x/y components into gradient magnitude.
3. Display all three magnitudes side by side.
4. Keep source and boundary handling fixed.

**Outputs**

Gradient comparison.

**Validation**

Record operator scaling; raw magnitudes need not be equally normalized.

**On failure**

Avoid attributing kernel gain differences to superior edge detection.

## 27. Inspect Border Effects on a Real Image

**Purpose**

Expose border choices on natural content.

**Inputs**

Same real image and filtering parameters.

**Procedure**

1. Apply a large averaging kernel with different border modes to the same real image.
2. Focus the comparison on boundary regions.
3. Save the diagnostic showing padding assumptions.

**Outputs**

Padding comparison figure.

**Validation**

Only border mode differs; inspect boundaries separately from interior.

**On failure**

Correct mismatched kernels before interpreting edge artifacts.

## 28. Filter RGB Images Safely

**Purpose**

Filter color without mixing channel indices.

**Inputs**

RGB image and spatial operator.

**Procedure**

1. Apply Gaussian filtering with sigma `(spatial_y, spatial_x, 0)` so channels are not mixed.
2. Apply bilateral filtering to the same RGB image.
3. Verify output shape and channel count are unchanged.
4. Save the color-filtering comparison.

**Outputs**

RGB filtered image.

**Validation**

Filter each channel spatially; preserve (height,width,3).

**On failure**

Do not smooth across the channel axis accidentally.

## 29. Demonstrate Numerical Safety

**Purpose**

Demonstrate numerical overflow and safe conversion.

**Inputs**

uint8 image and arithmetic example.

**Procedure**

1. Show how direct uint8 arithmetic can overflow or wrap.
2. Repeat the same operation in float or a wider type.
3. Clip to `[0,255]` only before casting back to uint8.
4. Use this rule for all signed intermediate computations.

**Outputs**

Unsafe/safe comparison.

**Validation**

Float computation precedes clipping and casting.

**On failure**

Replace overflowing arithmetic before using results downstream.

## 30. Define Filter-Selection Criteria

**Purpose**

Choose a filter from the corruption and preservation goal.

**Inputs**

Observed noise and preceding diagnostics.

**Procedure**

1. Use metric tables and edge-retention diagnostics to identify suitable filters for Gaussian versus impulse noise.
2. Keep the decision conditional on noise model and detail-preservation requirement.
3. Do not replace measured evidence with a generic ranking.

**Outputs**

Reasoned method-selection summary.

**Validation**

Tie each recommendation to shown experiments, not a universal ranking.

**On failure**

State uncertainty if clean reference or noise characterization is unavailable.

## 31. Implement a Reusable Spatial-Filtering Workflow

**Purpose**

State the reusable order of operations for a filtering experiment.

**Inputs**

Input role, selected method, boundary mode and parameters.

**Procedure**

1. Select a source image and document its noise model or filtering objective.
2. Convert to a safe computation dtype and choose the operator, boundary mode and parameters.
3. Apply the operator already demonstrated in the notebook.
4. Inspect the output and compute reference metrics only when a valid reference exists.
5. Record parameters, visual evidence and limits. This section specifies a workflow; it does not introduce a generic callable function.

**Outputs**

Documented workflow checklist in the main notebook.

**Validation**

Loading, floating arithmetic, filtering and evaluation follow the stated order.

**On failure**

Do not claim a reusable dispatcher function exists; the current section is a written workflow.

## 32. Run Final Validation

**Purpose**

Check convolution identities and generated diagnostics.

**Inputs**

Kernels, convolution results, metrics and files.

**Procedure**

1. Assert major filtered outputs match source dimensions.
2. Verify smoothing-kernel sums and expected zero-sum derivative/sharpening kernels.
3. Recheck manual convolution against SciPy.
4. Verify RGB outputs, intensity ranges, and all 18 required files.
5. Print the final pass message only after all checks succeed.

**Outputs**

Final validation outcome.

**Validation**

Kernel normalization/separability and manual/reference checks pass; required files exist.

**On failure**

Report failed invariant or missing output; figure presence alone does not establish denoising quality.

## Complete End-to-End Pseudocode

This is control-flow pseudocode, not an additional Python API. It follows the current implementation; review requirements and known gaps are distinguished below.

```text
resolve clean/noisy roles; load grayscale and RGB sources
create toy image and asymmetric kernel
compare manual correlation with convolution using a flipped kernel
implement direct convolution with symmetric padding and odd support
compare against SciPy with equivalent boundaries and declared allclose tolerances
vary border mode on the same toy input
apply mean filters; sweep support size
build normalized Gaussian kernels; verify the 1-D outer-product equivalence
sweep sigma and compare mean/Gaussian results on the same source
inspect noise models; apply median and bilateral filters
sweep median windows and bilateral parameters separately
for each Gaussian/impulse noisy input and matched clean reference:
    apply the compared filters with recorded settings
    compute floating MAE/MSE/PSNR; allow infinite PSNR at zero error
compare gradient summaries while noting noise also increases gradients
compute signed Laplacian sharpening and unsharp/high-boost outputs
compute Gx, Gy, magnitude and atan2 orientation; compare derivative operators
compare border effects on a real image
filter RGB channels independently; demonstrate overflow-safe arithmetic
summarize method choices and the standard filtering workflow
run implemented numerical checks and verify the required figure list
```

## Task Coverage

Every task defined in the Problem Statement is covered by the coding sequence above:

- **Task 1:** Data and Output Paths
- **Task 2:** Spatial Filtering Formulation
- **Task 3:** Kernel Properties and Normalization
- **Task 4:** Correlation vs Convolution
- **Task 5:** Direct Convolution Implementation
- **Task 6:** Border Handling
- **Task 7:** Mean / Box Filtering
- **Task 8:** Gaussian Filtering
- **Task 9:** Noise-Model Dependence
- **Task 10:** Median Filtering
- **Task 11:** Bilateral Filtering
- **Task 12:** Quantitative Denoising Metrics
- **Task 13:** Edge Preservation as a Secondary Check
- **Task 14:** Sharpening with the Laplacian
- **Task 15:** Unsharp Masking and High-Boost Filtering
- **Task 16:** First Derivatives and Image Gradients
- **Task 17:** Sobel vs Prewitt vs Scharr
- **Task 18:** Border Effects on a Real Image
- **Task 19:** Filtering RGB Images
- **Task 20:** Numerical Safety
- **Task 21:** Filter Selection Criteria
- **Task 22:** Standard Spatial-Filtering Workflow
- **Task 23:** Validation Checks


## Notebook Relationship

- `problem_statement.ipynb` defines **what must be solved**.
- `theory.ipynb` explains **why the methods work**.
- `requirements.ipynb` defines **what is needed to run the lab**.
- `algorithm.ipynb` defines **how to build the solution step by step**.
- `main.ipynb` is the **executable implementation**.


## Required Output Contract

The current `REQUIRED_OUTPUTS` list in `main.ipynb` contains 18 files under `../outputs/figures/`:

- `01_border_modes.png`
- `02_mean_filter.png`
- `03_mean_kernel_sizes.png`
- `04_gaussian_filter.png`
- `05_gaussian_sigma_sweep.png`
- `06_mean_vs_gaussian.png`
- `07_noise_types.png`
- `08_median_filter.png`
- `09_median_sizes.png`
- `10_bilateral_filter.png`
- `11_bilateral_parameter_sweep.png`
- `12_laplacian_sharpening.png`
- `13_unsharp_highboost.png`
- `14_sobel_gradients.png`
- `15_gradient_orientation.png`
- `16_derivative_operators.png`
- `17_padding_real_image.png`
- `18_color_filtering.png`

A presence check detects missing artifacts, not stale artifacts. Inspect the figures from the run being reported.

## Implementation Traceability

Reviewed against [source revision 66078c2](https://github.com/denoskume/msc-coro-dassip-portfolio/tree/66078c2481115d0b2596c6926e0750d136ba1005/labs/image-processing/filtering-in-spatial-domain/notebooks).

The following helpers/classes already exist in `main.ipynb`; algorithm operations that are not listed here are inline notebook cells or descriptive pseudocode:

- `locate_lab_root`
- `select_input_image`
- `load_grayscale_image`
- `load_rgb_image`
- `convolve_image_manually`
- `build_gaussian_kernel_2d`
- `build_gaussian_kernel_1d`
- `mean_absolute_error`
- `mean_squared_error`
- `root_mean_squared_error`
- `peak_signal_to_noise_ratio`
- `build_metric_summary`
- `compute_sobel_gradient_magnitude`
- `apply_spatial_filter`

Examples of **existing runtime assertions**, copied from the implementation:

```python
assert DATA_FILES
assert np.isclose(manual_corr, scipy_corr[2, 2])
assert np.isclose(manual_conv, scipy_conv[2, 2])
assert kh % 2 == 1 and kw % 2 == 1
assert np.allclose(manual_convolution_result, scipy_convolution_result, rtol=1e-10, atol=1e-12)
assert size % 2 == 1
assert sigma > 0
assert np.allclose(g5, outer)
assert workflow_result.shape == einstein_gaussian_noise_image.shape
assert np.all(np.isfinite(workflow_result))
```

This excerpt is not a new test suite and does not list every runtime exception.

## Implementation Limits and Interpretation

- NumPy symmetric padding in the manual example is compared with the matching SciPy reflect convention. Boundary conventions must agree for numerical comparisons.
- Denoising scores require the correct clean/noisy pair. Gradient magnitude can increase with noise and is not a standalone quality score.
- The final workflow section is a written sequence/checklist in the current main notebook, not an implemented generic dispatcher. No new filtering API is introduced here.

## 33. End

The coding plan ends after the implemented checks and required artifacts have been reviewed. Interpret outputs beside their corresponding experiment, then write the overall analysis at the bottom of `main.ipynb`. Report the implementation limits below; a numerical pass alone is not proof of physical validity or application-level accuracy.